# Car Body Damage Inspector

In Colab, choose a GPU runtime and run the cells from top to bottom. You’ll train two versions of the model and check how well each one outlines damage.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pycocotools'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime → Change runtime type → GPU, reconnect and run again.')
print('GPU:', torch.cuda.get_device_name(0))

## Set up the model

Use Mask R-CNN to find dents, scratches and cracks. Keep the labels in the same order as the Streamlit app so it shows the right name for each prediction.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from pycocotools.coco import COCO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor

CLASSES = ["background", "dent", "scratch", "crack"]


def make_model(pretrained=False):
    model = maskrcnn_resnet50_fpn(weights="DEFAULT" if pretrained else None,
                                  weights_backbone=None, min_size=640, max_size=960)
    # Set up the model to predict the damage labels used in the app.
    heads = model.roi_heads
    heads.box_predictor = FastRCNNPredictor(heads.box_predictor.cls_score.in_features, len(CLASSES))
    heads.mask_predictor = MaskRCNNPredictor(heads.mask_predictor.conv5_mask.in_channels, 256, len(CLASSES))
    return model




## Pick the training and test photos

Download 40 photos with labelled damage. Use 32 for training and set aside the other eight to check both versions later.

In [ ]:
import json, random
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
from PIL import ImageDraw
from tqdm.auto import tqdm
repo = 'DrBimmer/car-parts-and-damage-dataset'
revision = '2dbf46f7e23eb0d6f551d38e4f77a11fe9d045b9'
prefix = 'Car parts dataset/File1/'
paths = set(HfApi().list_repo_files(repo, repo_type='dataset', revision=revision))
annotations = sorted(p for p in paths if p.startswith(prefix+'ann/') and p.endswith('.json'))
random.Random(42).shuffle(annotations)
labels_by_name = {'dent': 1, 'scratch': 2, 'cracked': 3}

def fetch(name):
    return hf_hub_download(repo_id=repo, filename=name, repo_type='dataset', revision=revision)

def read_example(path):
    doc = json.loads(Path(fetch(path)).read_text())
    objects = [o for o in doc['objects'] if o['classTitle'].strip().lower() in labels_by_name]
    if not objects:
        return None
    name = Path(path).name.removesuffix('.json')
    with Image.open(fetch(prefix+'img/'+name)) as src:
        photo = src.convert('RGB')
    w, h = photo.size
    if (doc['size']['width'], doc['size']['height']) != (w,h):
        raise ValueError('Annotation dimensions differ for '+name)
    photo.thumbnail((720,720))
    masks, boxes, labels = [], [], []
    for obj in objects:
        poly = obj['points']
        if len(poly['exterior']) < 3:
            continue
        layer = Image.new('L',(w,h))
        draw = ImageDraw.Draw(layer)
        draw.polygon([tuple(p) for p in poly['exterior']], fill=1)
        for hole in poly.get('interior',[]):
            if len(hole)>=3:
                draw.polygon([tuple(p) for p in hole],fill=0)
        layer = layer.resize(photo.size, Image.Resampling.NEAREST)
        box = layer.getbbox()
        if box:
            masks.append(torch.from_numpy(np.array(layer)))
            boxes.append(box)
            labels.append(labels_by_name[obj['classTitle'].strip().lower()])
    if masks:
        return photo, dict(masks=torch.stack(masks),boxes=torch.tensor(boxes,dtype=torch.float32),
                           labels=torch.tensor(labels,dtype=torch.int64))

examples = []
for path in tqdm(annotations, desc='Reading annotations'):
    example = read_example(path)
    if example is not None:
        examples.append(example)
    if len(examples) == 40:
        break
assert len(examples)==40, f'Found only {len(examples)} usable examples'
training, held_out = examples[:32], examples[32:]
print('Training photos:',len(training),' Held-out photos:',len(held_out))
print('Training objects:', Counter(CLASSES[i] for _,t in training for i in t['labels'].tolist()))

## Train the first model

Run this cell to train on the 32 photos twice. After each pass, it saves the model as `damage.pt`.

In [ ]:
torch.manual_seed(42)
model = make_model(pretrained=True).cuda().train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0005)
checkpoint = Path('/content/damage.pt')
for epoch in range(2):
    running = 0.
    for idx in tqdm(torch.randperm(len(training)).tolist(),desc=f'Epoch {epoch+1}/2'):
        photo, target = training[idx]
        losses = model([to_tensor(photo).cuda()], [{k:v.cuda() for k,v in target.items()}])
        loss = sum(losses.values())
        if not torch.isfinite(loss):
            raise ValueError('Training loss became non-finite')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    torch.save({'model':model.state_dict(),'classes':CLASSES}, checkpoint.with_suffix('.tmp'))
    checkpoint.with_suffix('.tmp').replace(checkpoint)
    print(f'Epoch {epoch+1}: mean loss {running/len(training):.4f}; saved {checkpoint}')

## Add photos with cracks

The first 32 photos had no labelled cracks. Add 30 photos that show cracks to bring the training set to 62. Keep the same eight photos for testing.

In [ ]:
import hashlib

def photo_id(photo):
    return photo.size, hashlib.sha256(photo.tobytes()).digest()

# Skip photos already used for training or testing.
seen = {photo_id(photo) for photo, _ in training + held_out}
training_v2 = list(training)
for path in tqdm(annotations, desc='Finding crack photos'):
    doc = json.loads(Path(fetch(path)).read_text())
    if not any(obj['classTitle'].strip().lower() == 'cracked' for obj in doc['objects']):
        continue
    example = read_example(path)
    if example is None:
        continue
    photo, target = example
    key = photo_id(photo)
    if key in seen or not (target['labels'] == CLASSES.index('crack')).any().item():
        continue
    training_v2.append(example)
    seen.add(key)
    if len(training_v2) == 62:
        break
assert len(training_v2) == 62 and len(held_out) == 8
print('V2 training photos:', len(training_v2))

## Train the second model

Load the first model and train it for three more passes on all 62 photos. This cell saves the result as `damage_v2.pt`.

In [ ]:
torch.manual_seed(42)
model.load_state_dict(torch.load('/content/damage.pt', map_location='cpu', weights_only=True)['model'])
model.cuda().train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0005)
checkpoint = Path('/content/damage_v2.pt')
for epoch in range(3):
    running = 0.
    for idx in tqdm(torch.randperm(len(training_v2)).tolist(),desc=f'Epoch {epoch+1}/3'):
        photo, target = training_v2[idx]
        losses = model([to_tensor(photo).cuda()], [{k:v.cuda() for k,v in target.items()}])
        loss = sum(losses.values())
        if not torch.isfinite(loss):
            raise ValueError('Training loss became non-finite')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    torch.save({'model':model.state_dict(),'classes':CLASSES}, checkpoint.with_suffix('.tmp'))
    checkpoint.with_suffix('.tmp').replace(checkpoint)
    print(f'Epoch {epoch+1}: mean loss {running/len(training_v2):.4f}; saved {checkpoint}')

## Check one photo

Look at the orange outlines on the left to see where the damage was labelled. Compare them with the green outlines on the right to see what the second model found at a confidence threshold of 0.30.

In [ ]:
import cv2
import matplotlib.pyplot as plt
photo, target = held_out[0]
model.eval()
with torch.inference_mode():
    pred = model([to_tensor(photo).cuda()])[0]
truth, guess = np.array(photo), np.array(photo)
def outline(canvas, mask, color):
    contours, _ = cv2.findContours(mask.astype('uint8'),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas,contours,-1,color,2)
for mask in target['masks']:
    outline(truth,mask.numpy(),(255,140,0))
for score, mask, label in zip(pred['scores'],pred['masks'],pred['labels']):
    if score.item() >= 0.30:
        outline(guess,mask[0].cpu().numpy()>=0.5,(20,220,40))
        print(CLASSES[label.item()],round(score.item(),3))
fig, axes=plt.subplots(1,2,figsize=(14,6))
for ax,pic,title in zip(axes,[truth,guess],['Labelled damage','Predicted damage']):
    ax.imshow(pic); ax.set_title(title); ax.axis('off')
plt.show()

## Test both models on the same eight photos

Try confidence thresholds of 0.10 and 0.25. Count a prediction as correct when the damage type matches and the two masks have an intersection over union of at least 0.50. Each labelled defect can count once.

In [ ]:
@torch.inference_mode()
def evaluate(path, threshold):
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True)['model'])
    model.eval()
    device = next(model.parameters()).device
    correct = extra = missed = 0
    for photo, target in held_out:
        pred = model([to_tensor(photo).to(device)])[0]
        masks = target['masks'].to(device).bool()
        labels = target['labels'].to(device)
        matched = torch.zeros_like(labels, dtype=torch.bool)
        for i in pred['scores'].argsort(descending=True).tolist():
            if pred['scores'][i].item() < threshold:
                break
            mask = pred['masks'][i, 0] >= 0.5
            if not mask.any().item():
                continue
            candidates = torch.where((labels == pred['labels'][i]) & ~matched)[0]
            if len(candidates):
                actual = masks[candidates]
                # Divide the shared area by the total area covered by either mask.
                overlap = (actual & mask).sum((1, 2)) / (actual | mask).sum((1, 2))
                best = overlap.argmax()
                if overlap[best].item() >= 0.50:
                    matched[candidates[best]] = True
                    correct += 1
                    continue
            extra += 1
        missed += (~matched).sum().item()
    return correct, extra, missed

import pandas as pd
rows = []
for version, path in [('V1','/content/damage.pt'), ('V2','/content/damage_v2.pt')]:
    for threshold in (0.10, 0.25):
        tp, fp, fn = evaluate(path, threshold)
        precision = tp/(tp+fp) if tp+fp else None
        rows.append([version, threshold, tp, fp, fn, precision, tp/(tp+fn)])
pd.DataFrame(rows, columns=['Model','Threshold','Correct','Extra','Missed','Precision','Recall'])

## Results from my first comparison

| Model | Confidence threshold | Correct outlines | Extra guesses | Missed defects |
| :--- | ---: | ---: | ---: | ---: |
| V1 | 0.10 | 0 | 43 | 47 |
| V1 | 0.25 | 0 | 0 | 47 |
| V2 | 0.10 | 12 | 556 | 35 |
| V2 | 0.25 | 5 | 24 | 42 |

I recorded these numbers in my original Colab run. When you run the notebook, the cell above shows your results.

## Save the model for the app

Download `damage_v2.pt`. Put a copy in the folder containing `app.py` and rename the copy to `damage.pt`.

In [ ]:
from google.colab import files
files.download('/content/damage_v2.pt')